# Dataplex Universal Catalog - Many-to-Many (N:N) Relationship Validation

This notebook illustrates how to model Many-to-Many relationships in Google Cloud Dataplex Universal Catalog (e.g., Topics distributed across multiple Clusters).

It provisions the necessary Entry Group and Entry Types, instantiates the assets, and establishes bi-directional links without incurring N+1 operational discovery query degradation.

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("Successfully authenticated with Google Cloud.")

In [ ]:
import json
import time
import google.auth
from google.auth.transport.requests import AuthorizedSession

# ==============================================================================
# CONFIGURATION
# ==============================================================================
PROJECT_ID = "bipin-dev-project-465919" #@param {type:"string"}
LOCATION = "us-central1"
ENTRY_GROUP_ID = "kafka-group"

BASE_URL = "https://dataplex.googleapis.com/v1"

def get_authenticated_session():
    credentials, project = google.auth.default()
    return AuthorizedSession(credentials)

print(f"Configured targeting Project: {PROJECT_ID}")

In [ ]:
def ensure_entry_group_exists(session, project_id, location, entry_group_id):
    name = f"projects/{project_id}/locations/{location}/entryGroups/{entry_group_id}"
    get_url = f"{BASE_URL}/{name}"

    print(f"Checking if Entry Group exists: {entry_group_id}...")
    response = session.get(get_url)

    if response.status_code == 200:
        print(f"Entry Group {entry_group_id} already exists.")
        return name
    elif response.status_code == 404:
        print(f"Entry Group {entry_group_id} not found. Creating...")
        create_url = f"{BASE_URL}/projects/{project_id}/locations/{location}/entryGroups"
        headers = {"Content-Type": "application/json"}
        params = {"entryGroupId": entry_group_id}
        body = {
            "displayName": "Kafka Assets Group",
            "description": "Central catalog for tracking Kafka Topics and Clusters",
        }

        create_response = session.post(create_url, headers=headers, params=params, json=body)

        if create_response.status_code in [200, 201, 202]:
            max_retries = 12
            for attempt in range(max_retries):
                time.sleep(5)
                poll_response = session.get(get_url)
                if poll_response.status_code == 200:
                    return name
            raise Exception(f"Entry Group {entry_group_id} did not become active.")
        else:
            raise Exception("Entry Group creation failed")

def ensure_entry_type_exists(session, project_id, location, entry_type_id):
    name = f"projects/{project_id}/locations/{location}/entryTypes/{entry_type_id}"
    get_url = f"{BASE_URL}/{name}"

    response = session.get(get_url)
    if response.status_code == 200:
        return name
    elif response.status_code == 404:
        create_url = f"{BASE_URL}/projects/{project_id}/locations/{location}/entryTypes"
        headers = {"Content-Type": "application/json"}
        params = {"entryTypeId": entry_type_id}
        body = {
            "displayName": f"Kafka {entry_type_id.capitalize()} Type",
            "platform": "Kafka",
            "system": "Kafka",
        }

        create_response = session.post(create_url, headers=headers, params=params, json=body)

        if create_response.status_code in [200, 201, 202]:
            for attempt in range(12):
                time.sleep(5)
                if session.get(get_url).status_code == 200:
                    return name
            raise Exception(f"Entry Type {entry_type_id} timed out.")

def ensure_entry_exists(session, project_id, location, entry_group_id, entry_id, entry_type_id):
    name = f"projects/{project_id}/locations/{location}/entryGroups/{entry_group_id}/entries/{entry_id}"
    get_url = f"{BASE_URL}/{name}"

    response = session.get(get_url)
    if response.status_code == 200:
        return name
    elif response.status_code == 404:
        create_url = f"{BASE_URL}/projects/{project_id}/locations/{location}/entryGroups/{entry_group_id}/entries"
        headers = {"Content-Type": "application/json"}
        params = {"entryId": entry_id}
        body = {
            "entryType": f"projects/{project_id}/locations/{location}/entryTypes/{entry_type_id}",
            "aspects": {},
        }

        create_response = session.post(create_url, headers=headers, params=params, json=body)
        if create_response.status_code in [200, 201]:
            return name
        else:
            raise Exception(f"Entry creation failed for {entry_id}")

def create_entry_link(session, project_id, location, entry_group_id, source_entry, target_entry):
    url = f"{BASE_URL}/projects/{project_id}/locations/{location}/entryGroups/{entry_group_id}/entryLinks"
    headers = {"Content-Type": "application/json"}
    entry_link_id = f"link-{source_entry.split('/')[-1]}-{target_entry.split('/')[-1]}"
    params = {"entryLinkId": entry_link_id}

    body = {
        "entryLinkType": "projects/dataplex-types/locations/global/entryLinkTypes/related",
        "entryReferences": [
            {"name": source_entry, "type": "UNSPECIFIED"},
            {"name": target_entry, "type": "UNSPECIFIED"},
        ],
    }

    response = session.post(url, headers=headers, params=params, json=body)
    if response.status_code in [200, 201]:
        print(f"Created Entry Link: {entry_link_id}")
    elif response.status_code != 409:
        print(f"Failed to create Entry Link {entry_link_id}. Status: {response.status_code}")
        print("Details:", response.text)

def lookup_entry_links(session, project_id, location, entry_name):
    url = f"{BASE_URL}/projects/{project_id}/locations/{location}:lookupEntryLinks"
    params = {
        "entry": entry_name,
        "entryLinkTypes": "projects/dataplex-types/locations/global/entryLinkTypes/related",
        "entryMode": "ENTRY_MODE_UNSPECIFIED",
    }
    response = session.get(url, params=params)
    if response.status_code == 200:
        return response.json().get("entryLinks", [])
    return []

In [ ]:
session = get_authenticated_session()

# 1. Ensure Parent resources
ensure_entry_group_exists(session, PROJECT_ID, LOCATION, ENTRY_GROUP_ID)
ensure_entry_type_exists(session, PROJECT_ID, LOCATION, "kafka-topic")
ensure_entry_type_exists(session, PROJECT_ID, LOCATION, "kafka-cluster")

# 2. Create 2 Clusters
cluster_entries = []
for i in range(1, 3):
    name = ensure_entry_exists(session, PROJECT_ID, LOCATION, ENTRY_GROUP_ID, f"cluster-{i}", "kafka-cluster")
    cluster_entries.append(name)
print("Clusters ensured.")

# 3. Create 10 Topics
topic_entries = []
for i in range(1, 11):
    name = ensure_entry_exists(session, PROJECT_ID, LOCATION, ENTRY_GROUP_ID, f"topic-{i}", "kafka-topic")
    topic_entries.append(name)
print("Topics ensured.")

# 4. Link Topics to Clusters
for i, topic_entry in enumerate(topic_entries):
    create_entry_link(session, PROJECT_ID, LOCATION, ENTRY_GROUP_ID, topic_entry, cluster_entries[i % 2])
    create_entry_link(session, PROJECT_ID, LOCATION, ENTRY_GROUP_ID, topic_entry, cluster_entries[(i + 1) % 2])

print("\nMany-to-Many Links generated.")

In [ ]:
def find_clusters_for_topic(topic_id):
    entry_name = f"projects/{PROJECT_ID}/locations/{LOCATION}/entryGroups/{ENTRY_GROUP_ID}/entries/{topic_id}"
    links = lookup_entry_links(session, PROJECT_ID, LOCATION, entry_name)
    clusters = [ref.get("name").split("/")[-1] for link in links for ref in link.get("entryReferences", []) if topic_id not in ref.get("name")]
    print(f"Topic [{topic_id}] is linked to Clusters: {list(set(clusters))}")

def find_topics_in_cluster(cluster_id):
    entry_name = f"projects/{PROJECT_ID}/locations/{LOCATION}/entryGroups/{ENTRY_GROUP_ID}/entries/{cluster_id}"
    links = lookup_entry_links(session, PROJECT_ID, LOCATION, entry_name)
    topics = [ref.get("name").split("/")[-1] for link in links for ref in link.get("entryReferences", []) if cluster_id not in ref.get("name")]
    print(f"Cluster [{cluster_id}] hosts Topics: {list(set(topics))}")

find_clusters_for_topic("topic-1")
find_topics_in_cluster("cluster-1")

In [ ]:
#@title Cleanup Dataplex Entries

def cleanup_entries():
    session = get_authenticated_session()
    list_url = f"{BASE_URL}/projects/{PROJECT_ID}/locations/{LOCATION}/entryGroups/{ENTRY_GROUP_ID}/entries"
    
    print(f"Listing entries in group: {ENTRY_GROUP_ID}...")
    response = session.get(list_url)
    
    if response.status_code != 200:
        print(f"Failed to list entries. Status: {response.status_code}")
        print(response.text)
        return

    entries = response.json().get("entries", [])
    if not entries:
        print("No entries found to clean up.")
        return

    print(f"Found {len(entries)} entries. Starting deletion...")
    for entry in entries:
        entry_name = entry.get("name")
        entry_id = entry_name.split("/")[-1]
        if entry_id == "kafka-group_entry": # Skip the group entry itself if returned
            continue
            
        print(f"Deleting entry: {entry_id}...")
        delete_url = f"{BASE_URL}/{entry_name}"
        del_response = session.delete(delete_url)
        
        if del_response.status_code in [200, 204]:
            print(f"Successfully deleted: {entry_id}")
        else:
            print(f"Failed to delete {entry_id}. Status: {del_response.status_code}")

cleanup_entries()